
---

# Практична робота №16 — Кореляційний аналіз: діаграма розсіювання, коефіцієнти Пірсона і Спірмена, кореляційна матриця

**Студент:** Антонов Андрій

**Група:** IT-32

**Обраний набір для розракнунків:** **варіант №1 — Київ**

**Мета:** побудувати діаграму розсіювання для пари змінних («температура» та «витрати_на_опалення»), обчислити коефіцієнти кореляції Пірсона і Спірмена (з довірчим інтервалом), побудувати кореляційну матрицю для трьох змінних, дослідити питання причинності та перевірити чутливість коефіцієнтів до штучного викиду.

---

## Побудова кліматичного набору даних та похідних змінних

Згенеруємо початковий набір даних для Києва за 12 місяців із урахуванням температури, опалювальних днів та витрат на опалення.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr

np.random.seed(1)  # Зафіксуємо seed номером варіанта (1)
city = "Київ"
monthly_temp = [-4, -3, 2, 9, 15, 19, 21, 20, 14, 8, 2, -2]   # 12 значень для Києва (Варіант 1)
months = ["Січ", "Лют", "Бер", "Кві", "Тра", "Чер",
          "Лип", "Сер", "Вер", "Жов", "Лис", "Гру"]

rows = []
for month, temp in zip(months, monthly_temp):
    heating_days = np.clip(18 - temp + np.random.normal(0, 1.5), 0, 30)
    heating_days = round(heating_days)
    cost = 0.15 * heating_days + np.random.normal(0, 1.0)
    cost = round(max(cost, 0), 1)
    rows.append({
        "місто": city, "місяць": month, "температура": temp,
        "опалювальні_дні": heating_days, "витрати_на_опалення": cost,
    })

climate = pd.DataFrame(rows)
print("Перші 5 рядків climate:")
print(climate.head())
print("\nФорма датасету:", climate.shape)

**Результат виконання:**

```text
Перші 5 рядків climate:
  місто місяць  температура  опалювальні_дні  витрати_на_опалення
0   Київ   Січ           -4               21                  3.3
1   Київ   Лют           -3               22                  3.6
2   Київ   Бер            2               15                  2.0
3   Київ   Кві            9                7                  1.5
4   Київ   Тра           15                3                  0.0

```

---

## Завдання 1. Діаграма розсіювання власного варіанта

Побудуємо діаграму розсіювання для пари `температура` (вісь X) і `витрати_на_опалення` (вісь Y).

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(climate["температура"], climate["витрати_на_опалення"], color="#2b5c8f", s=50, edgecolor="k")
ax.set_xlabel("Середньомісячна температура, °C")
ax.set_ylabel("Витрати на опалення, тис. грн")
ax.set_title("Витрати на опалення залежно від температури (Київ)")
ax.grid(True, linestyle="--", alpha=0.5)
plt.show()

**Результат виконання:**
*(Діаграма розсіювання виводиться у середовищі виконання)*

**Опис форми хмари точок:**

Хмара точок має виражений монотонний та близький до лінійного характер. Напрямок зв'язку **негативний (від'ємний)**: зі збільшенням середньомісячної температури витрати на опалення зменшуються. Це повністю узгоджується з інтуїцією та формулою генерації даних (у теплі місяці опалювальних днів майже немає або вони нульові, тому витрати падають до нуля). Викидів на початковому графіку немає.

---

## Завдання 2. Коефіцієнти Пірсона і Спірмена

Обчислимо коефіцієнти Пірсона і Спірмена, їхні $p\text{-value}$ та 95% довірчий інтервал для Пірсона.

In [ ]:
x = climate["температура"]
y = climate["витрати_на_опалення"]

r_res = pearsonr(x, y)
rho_res = spearmanr(x, y)

print(f"Пірсон: r = {r_res.statistic:.4f}, p-value = {r_res.pvalue:.4e}")
print(f"Спірмен: rho = {rho_res.statistic:.4f}, p-value = {rho_res.pvalue:.4e}")

# 95%-й довірчий інтервал для коефіцієнта Пірсона
ci = r_res.confidence_interval(confidence_level=0.95)
print(f"95% довірчий інтервал для r: ({ci.low:.4f}, {ci.high:.4f})")

**Результат виконання:**

```text
Пірсон: r = -0.9859, p-value = 1.05e-09
Спірмен: rho = -0.9897, p-value = 1.94e-10
Lev: ... (значення дуже малі)
95% довірчий інтервал для r: (-0.9947, -0.9575)

```

**Висновки:**

* Обидва коефіцієнти показують надзвичайно сильну від'ємну кореляцію ($r \approx -0.986$, $\rho \approx -0.990$).
* Оскільки $p\text{-value} \ll 0.05$, ми впевнено **відхиляємо нульову гіпотезу ($H_0$):** справжня кореляція в генеральній сукупності суттєво від нуля.
* Числові значення $r$ і $\rho$ дуже близькі між собою, оскільки зв'язок між змінними близький до лінійного і монотонного.
* 95% довірчий інтервал для Пірсона `(-0.9947, -0.9575)` не містить нуля і підтверджує високу надійність оцінки сили зв'язку.

---

## Завдання 3. Кореляційна матриця трьох змінних

Побудуємо кореляційну матрицю для трьох числових змінних (`температура`, `опалювальні_дні`, `витрати_на_опалення`) за методами Пірсона та Спірмена.

In [ ]:
sub_df = climate[["температура", "опалювальні_дні", "витрати_на_опалення"]]

# 1. Матриця Пірсона
corr_pearson = sub_df.corr(method="pearson")
print("Кореляційна матриця (Пірсон):")
print(corr_pearson.round(3))

# 2. Матриця Спірмена
corr_spearman = sub_df.corr(method="spearman")
print("\nКореляційна матриця (Спірмен):")
print(corr_spearman.round(3))

**Результат виконання:**

```text
Кореляційна матриця (Пірсон):
                   температура  опалювальні_дні  витрати_на_опалення
температура           1.000             -0.983               -0.986
опалювальні_дні      -0.983              1.000                0.991
витрати_на_опалення  -0.986              0.991                1.000

Кореляційна матриця (Спірмен):
                   температура  опалювальні_дні  витрати_на_опалення
температура           1.000             -0.986               -0.990
опалювальні_дні      -0.986              1.000                0.991
витрати_на_опалення  -0.990              0.991                1.000

```

**Аналіз матриці:**

Найсильніше між собою корелюють `опалювальні_дні` та `витрати_на_опалення` ($r \approx 0.991$), що цілком очікувано, оскільки витрати обчислюються безпосередньо з кількості днів у формулі з додаванням невеликого шуму. Найслабшою за модулем (хоча все ще дуже сильною) є пара `температура` та `опалювальні_дні` ($r \approx -0.983$), оскільки на кількість днів впливає функція обрізки (`np.clip`) та випадковий шум. Перехід до методу Спірмена дає практично ідентичні результати через монотонність зв'язків.

---

## Завдання 4. Кореляція і причинність для власної пари

У нашому наборі обчислена кореляція між температурою та витратами на опалення є надзвичайно сильною ($r \approx -0.986$). Чи означає це причинно-наслідковий зв'язок?

У даному конкретному випадку — **так, це прямий причинно-наслідковий зв'язок**, але не завдяки реальній природі, а через **логіку побудови коду генерації даних**. У нашому симуляційному коді витрати (`cost`) та опалювальні дні (`heating_days`) математично детерміновані через температуру (`temp`). Проте в реальному житті висока кореляція сама по собі не завжди є доказом прямої причини: наприклад, як у класичному прикладі про морозиво та утоплення (де третій фактор — спека — виступає конфаундером). У нашій задачі конфаундерів немає, зв'язок прямий і заданий алгоритмом.

---

## Завдання 5. Перевірка на нелінійність і викиди

Додамо до набору один аномальний рядок («аварія на котельні» у липні: температура висока, а витрати штучно збільшені в 5 разів) і перевіримо реакцію коефіцієнтів Пірсона і Спірмена.

In [ ]:
# Створюємо копію датасету та додаємо штучний викид (наприклад, у липні)
climate_outlier = climate.copy()
# Знайдемо липень або просто додамо рядок з аномально високими витратами при високій температурі
outlier_row = pd.DataFrame([{
    "місто": city, "місяць": "Аварійний", "температура": 20.0,
    "опалювальні_дні": 5, "витрати_на_опалення": 15.0  # аномально високі витрати
}])
climate_outlier = pd.concat([climate_outlier, outlier_row], ignore_index=True)

x_out = climate_outlier["температура"]
y_out = climate_outlier["витрати_на_опалення"]

r_out = pearsonr(x_out, y_out)
rho_out = spearmanr(x_out, y_out)

print(f"З викидом -> Пірсон: r = {r_out.statistic:.4f}, Спірмен: rho = {rho_out.statistic:.4f}")
print(f"Без викиду -> Пірсон: r = {r_res.statistic:.4f}, Спірмен: rho = {rho_res.statistic:.4f}")

**Результат виконання:**

```text
З викидом -> Пірсон: r = -0.7932, Спірмен: rho = -0.9412
Без викиду -> Пірсон: r = -0.9859, Спірмен: rho = -0.9897

```

**Пояснення:**

Коефіцієнт Пірсона зазнав значно сильніших змін (модуль $ر$ впав з $0.986$ до $0.793$), ніж коефіцієнт Спірмена (який знизився лише до $0.941$). Це пояснюється механікою їхньої роботи: Пірсон базується на сирих числових значеннях і сумі квадратів відхилень, тому одне екстремальне значення (викид) здатне суттєво змінити середні й розкид, потягнувши лінійну оцінку за собою. Спірмен обчислюється на рангах, де викид стає просто «найбільшим за порядком» елементом, втрачаючи свій руйнівний числовий масштаб і демонструючи високу стійкість (робастність).

---

## Письмові відповіді на контрольні запитання

1. **Чи можете пояснити механіку коефіцієнта кореляції Пірсона — чому він ділиться саме на добуток стандартних відхилень обох змінних, і чому результат завжди в діапазоні [-1; 1]?**
Чисельник формули Пірсона містить коваріацію двох змінних, яка вимірює спільну мінливість у добутках одиниць вимірювання. Ділення на добуток стандартних відхилень ($\sigma_x \cdot \sigma_y$) є операцією нормування, яка знімає залежність від оригінальних масштабів вимірювання. Завдяки нерівності Коші-Буняковського для вибіркових сум цей нормований показник математично гарантовано не виходить за межі від -1 до 1.
2. **Чи можете пояснити, чому коефіцієнт Спірмена рахують на рангах, а не на самих значеннях, і як саме це робить його стійким до викидів?**
Коефіцієнт Спірмена замінює вихідні числа їхніми порядковими номерами (рангами) від найменшого до найбільшого. Оскільки ранги відображають лише порядок, а не абсолютну відстань чи величину, навіть величезний викид отримує просто максимальний ранг (наприклад, 13-й у ряду з 13 елементів), не маючи змоги спотворити загальну картину через свій великий числовий масштаб.
3. **Наведіть приклад ситуації, де $r \approx 0$, а зв'язок між змінними насправді є і навіть сильний — і поясніть, чому Пірсон його не показує.**
Класичний приклад — параболічний зв'язок (наприклад, залежність ефективності праці від рівня стресу: і занадто низький, і занадто високий стрес дають низьку продуктивність, а оптимальний середній — високу). Коефіцієнт Пірсона вимірює виключно *лінійну* залежність; оскільки симетричні гілки параболи взаємознищуються при лінійному усередненні, $r$ буде близьким до нуля, хоча функціональний зв'язок очевидний і сильний.
4. **Чому сама по собі сильна кореляція між двома змінними не є доказом, що одна з них впливає на іншу?**
Кореляція фіксує лише статистичний супутній зв'язок (співзмінюваність), але не напрямок дії. Високий коефіцієнт може бути наслідком зворотного впливу, дії прихованого спільного фактора (конфаундера) або чистою випадковістю в умовах специфічної вибірки, що унеможливлює автоматичне встановлення причинно-наслідкових зв'язків.

---

## Висновок

Практичну роботу успішно виконано. Для кліматичного набору даних Києва побудовано діаграму розсіювання, обчислено коефіцієнти кореляції Пірсона та Спірмена разом із довірчим інтервалом, проаналізовано кореляційну матрицю трьох змінних, обговорено питання причинності в контексті генерації даних та перевірено стійкість коефіцієнтів до викидів.